# 04 — Zero-shot, one-shot, few-shot

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase02/notebooks/04_prompting_techniques.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivo.** Comparar las tres técnicas en una tarea concreta: clasificación de sentimiento. Vas a ver cómo unos pocos ejemplos cambian el comportamiento.


In [ ]:
%pip install --quiet groq


In [ ]:
import os
from groq import Groq

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    assert os.environ.get("GROQ_API_KEY"), "Exportá GROQ_API_KEY."

client = Groq()
MODELS = {
    "llama_fast":   "llama-3.1-8b-instant",
    "llama_strong": "llama-3.3-70b-versatile",
    "qwen_reason":  "qwen/qwen3-32b",
    "deepseek":     "deepseek-r1-distill-llama-70b",
    "gemma":        "gemma2-9b-it",
}
MODEL = MODELS["llama_strong"]  # cambiá la clave para probar otros modelos

def clasificar(messages, temperature=0.1):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


## Reseñas de prueba

Mezclamos casos fáciles y ambiguos para ver dónde fallan las distintas técnicas.


In [ ]:
RESEÑAS = [
    "El servicio fue impecable, vuelvo seguro.",                                 # claramente positiva
    "Pésimo, no me devolvieron la plata.",                                       # claramente negativa
    "Está OK, nada del otro mundo pero cumple.",                                 # neutral
    "El servicio fue lento pero la comida estuvo buena.",                        # ambiguo
    "Llegó a tiempo, eso es lo único bueno que puedo decir.",                    # sarcástico/negativo
    "Hace lo que promete, ni más ni menos.",                                     # neutral
]


## Zero-shot

Solo le decimos qué hacer, sin ejemplos.


In [ ]:
SYSTEM_ZS = "Sos un clasificador de sentimientos. Respondés SOLO con una palabra: POSITIVA, NEUTRAL o NEGATIVA."

print("ZERO-SHOT")
print("-" * 60)
for r in RESEÑAS:
    out = clasificar([
        {"role": "system", "content": SYSTEM_ZS},
        {"role": "user", "content": f"Reseña: {r}"},
    ])
    print(f"  {out:<10} <- {r}")


## Few-shot

Le damos 3 ejemplos resueltos. Observá cómo mejora la consistencia en los casos ambiguos.


In [ ]:
SYSTEM_FS = "Sos un clasificador de sentimientos. Respondés SOLO con: POSITIVA, NEUTRAL o NEGATIVA."

EJEMPLOS = '''Reseña: Excelente atención y precio justo.
Etiqueta: POSITIVA

Reseña: Lo recibí roto y nadie responde.
Etiqueta: NEGATIVA

Reseña: Cumple, pero tarda más que la competencia.
Etiqueta: NEUTRAL

'''

print("FEW-SHOT")
print("-" * 60)
for r in RESEÑAS:
    user = EJEMPLOS + f"Reseña: {r}\nEtiqueta:"
    out = clasificar([
        {"role": "system", "content": SYSTEM_FS},
        {"role": "user", "content": user},
    ])
    print(f"  {out:<10} <- {r}")


## Parte 2 — Cuando zero-shot ya no alcanza

El sentimiento de reseñas es una tarea **fácil**: el modelo conoce las clases (positivo / negativo / neutral) sin que se las definas. Por eso zero-shot ya anda muy bien.

Probemos algo más realista: **triaje de tickets de soporte con un esquema propio**, donde:

- Las etiquetas son **idiosincrásicas** del producto (no las podés googlear).
- Hay **clases ambiguas** (un mismo ticket podría caer en dos).
- Necesitás un **formato de salida estricto** para alimentar un sistema downstream.


## La tarea

Un ticket entra y queremos clasificarlo en una de **6 etiquetas internas**:

| Etiqueta | Significa |
|---|---|
| `BUG_BLOQUEANTE` | El usuario no puede operar (login caído, pago no procesa, error 500). |
| `BUG_VISUAL` | Algo se ve mal pero la funcionalidad anda (CSS roto, ícono cortado). |
| `FEATURE_REQUEST` | Pide algo que no existe todavía. |
| `DUDA_DOCS` | Duda que se resuelve leyendo la doc del producto. |
| `DUDA_BILLING` | Duda sobre planes, facturación, precios, suscripción. |
| `OUT_OF_SCOPE` | No es nuestro problema (consulta sobre otro producto, spam, queja sin acción posible). |


In [ ]:
TICKETS = [
    "Hace 2 horas que no puedo loguearme, me tira 'session expired' en loop.",
    "El logo de la barra se ve cortado a la mitad en Safari mobile.",
    "¿Tienen plan anual con descuento o solo mensual?",
    "Necesitaría poder exportar el reporte a Excel, no solo a PDF.",
    "No entiendo cómo configurar el webhook, leí la doc pero quedé igual.",
    "¿Hola? Me podés pasar el teléfono de Movistar?",
    "El pago me lo cobró dos veces, urgente.",
    "Sería genial poder customizar los colores del dashboard.",
]


## Zero-shot

Le decimos al modelo qué tiene que hacer **en palabras**, sin ejemplos.


In [ ]:
SYSTEM_ZS = '''Sos un sistema de triaje de tickets de soporte.
Clasificá cada ticket en exactamente una de estas etiquetas:
BUG_BLOQUEANTE, BUG_VISUAL, FEATURE_REQUEST, DUDA_DOCS, DUDA_BILLING, OUT_OF_SCOPE.

Respondé SOLO con la etiqueta, sin explicación.'''

print("ZERO-SHOT")
print("-" * 70)
zero_shot_results = []
for t in TICKETS:
    out = clasificar([
        {"role": "system", "content": SYSTEM_ZS},
        {"role": "user", "content": t},
    ])
    zero_shot_results.append(out)
    print(f"  {out:<25} <- {t[:60]}")


**¿Qué pasa típicamente?**

- El modelo a veces inventa etiquetas (`BUG_CRITICO`, `bug_visual`, `Bug Bloqueante`) o las escribe distinto.
- Agrega texto extra ("Etiqueta: BUG_BLOQUEANTE", "Esta sería FEATURE_REQUEST porque...").
- En los casos ambiguos (consulta sobre billing vs duda general; bug visual vs feature request) elige distinto cada vez.

Eso pasa porque el modelo **no conoce tu esquema** — está adivinando.


## Few-shot

Le mostramos 5 ejemplos resueltos. Elegidos para cubrir los casos donde zero-shot suele fallar.


In [ ]:
EJEMPLOS_FS = '''
Ticket: No me carga el dashboard, queda cargando infinito.
Etiqueta: BUG_BLOQUEANTE

Ticket: El botón "Guardar" aparece cortado en pantalla chica.
Etiqueta: BUG_VISUAL

Ticket: ¿Cuánto sale el plan empresa?
Etiqueta: DUDA_BILLING

Ticket: Necesito una forma de duplicar los proyectos.
Etiqueta: FEATURE_REQUEST

Ticket: Disculpá, era para otra empresa.
Etiqueta: OUT_OF_SCOPE

'''

SYSTEM_FS = '''Sos un sistema de triaje de tickets de soporte.
Clasificá cada ticket en exactamente una de estas etiquetas:
BUG_BLOQUEANTE, BUG_VISUAL, FEATURE_REQUEST, DUDA_DOCS, DUDA_BILLING, OUT_OF_SCOPE.

Respondé SOLO con la etiqueta, sin explicación.'''

print("FEW-SHOT")
print("-" * 70)
few_shot_results = []
for t in TICKETS:
    user = EJEMPLOS_FS + f"Ticket: {t}\nEtiqueta:"
    out = clasificar([
        {"role": "system", "content": SYSTEM_FS},
        {"role": "user", "content": user},
    ])
    few_shot_results.append(out)
    print(f"  {out:<25} <- {t[:60]}")


## Comparación lado a lado


In [ ]:
print(f"{'TICKET':<60} {'ZERO-SHOT':<22} {'FEW-SHOT':<22}")
print("-" * 104)
for t, z, f in zip(TICKETS, zero_shot_results, few_shot_results):
    short = (t[:55] + '...') if len(t) > 58 else t
    flag = "  ⚠" if z.strip() != f.strip() else ""
    print(f"{short:<60} {z:<22} {f:<22}{flag}")


## El takeaway

El few-shot **no le enseña a clasificar** al modelo (Llama 3.3 ya entiende perfecto cada ticket). Lo que hace es:

1. **Fijar el formato exacto** — solo la etiqueta, sin texto extra, en MAYÚSCULAS exactas.
2. **Resolver ambigüedad de las clases** — qué cae en `DUDA_BILLING` vs `DUDA_DOCS`, qué es `BUG_VISUAL` vs `FEATURE_REQUEST`.
3. **Anclar el vocabulario** — el modelo no inventa `BUG_CRITICO` ni `bug_visual`.

Ese es el patrón general: **few-shot es para fijar formato y resolver tu esquema propio**, no para enseñarle al modelo lo que ya sabe.


## Para experimentar

- Cambiá los ejemplos del few-shot por casos más cercanos a tu dominio. Probablemente mejore más.
- Probá con 1 solo ejemplo (one-shot) y compará con few-shot.
- Pedí al modelo que devuelva además una **explicación** de por qué clasificó así. Esto es CoT, lo vemos en el próximo notebook.
- Probá con `temperature=0.7` en zero-shot — ¿se vuelve más inconsistente?
- Cambiá `MODEL` a `MODELS["llama_fast"]` (Llama 3.1 8B) y mirá cómo cae la calidad — los modelos chicos sufren más sin few-shot.
